In [9]:
!pip install roboflow

In [10]:
!pip install ultralytics

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
import json
import time
import random
import numpy as np
import shutil
from datetime import datetime
from ultralytics import YOLO
from roboflow import Roboflow
import glob

In [12]:
import yaml

yaml_data = {
    'train': '/kaggle/input/datasets/sujityp/rdd2022/datasets/train',
    'val': '/kaggle/input/datasets/sujityp/rdd2022/datasets/valid',
    'test': '/kaggle/input/datasets/sujityp/rdd2022/datasets/test',
    'nc': 5,
    'names': ['longitudinal crack', 'transverse crack', 'alligator crack', 'other corruption', 'pothole']
}

DATA_YAML = '/kaggle/working/data.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

EXPERIMENTS_DIR = "/kaggle/working/runs/detect"

val_images = glob.glob('/kaggle/input/**/valid/images/*.jpg', recursive=True) + \
             glob.glob('/kaggle/input/**/val/images/*.jpg', recursive=True)
DUMMY_IMAGE_PATH = val_images[0] if val_images else None

In [13]:
MODELS = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
EPOCHS = 15         
BATCH_SIZE = 64      # 
IMAGE_SIZE = 640
LEARNING_RATE = 0.01
OPTIMIZER = "auto"  
SEED = 42



In [14]:
def train_one_model(weights_name: str) -> dict:
    model_name = weights_name.replace(".pt", "")
    run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
 
    print(f"\n{'='*40}")
    print(f"🚀 Start training {model_name} 🚀")
    print(f"{'='*40}")
 
    # أ. التدريب
    model = YOLO(weights_name)
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        lr0=LEARNING_RATE,
        optimizer=OPTIMIZER,
        seed=SEED,
        project=EXPERIMENTS_DIR,
        name=run_name,
        device=0, # تفعيل كارت الشاشة
        exist_ok=True
    )
 
    # ب. استخراج أفضل نموذج
    best_weights_path = os.path.join(EXPERIMENTS_DIR, run_name, "weights", "best.pt")
    best_model = YOLO(best_weights_path)
    
    # ج. التقييم على بيانات التحقق (Validation)
    metrics = best_model.val(data=DATA_YAML)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map
    precision = metrics.box.mp
    recall = metrics.box.mr
 
    # د. حساب حجم النموذج بالميجابايت
    model_size_mb = os.path.getsize(best_weights_path) / (1024 * 1024)
    
    # هـ. حساب سرعة الاستجابة (FPS & Inference Time)
    fps = None
    avg_inference_ms = None
    
    if DUMMY_IMAGE_PATH and os.path.exists(DUMMY_IMAGE_PATH):
        # Warm-up run (لتجهيز كارت الشاشة)
        best_model.predict(DUMMY_IMAGE_PATH, verbose=False)
        
        start = time.time()
        n_runs = 20
        for _ in range(n_runs):
            best_model.predict(DUMMY_IMAGE_PATH, verbose=False)
        elapsed = time.time() - start
        
        avg_inference_ms = (elapsed / n_runs) * 1000
        fps = 1000 / avg_inference_ms
 
    # و. تجميع النتائج
    result = {
        "Model": model_name,
        "mAP50": round(map50, 4),
        "mAP50-95": round(map50_95, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "Size_MB": round(model_size_mb, 2),
        "Inference_ms": round(avg_inference_ms, 2) if avg_inference_ms else None,
        "FPS": round(fps, 2) if fps else None,
        "Weights_Path": best_weights_path
    }
 
    print(f"\n📊 Results for {model_name}:")
    print(f"mAP50: {result['mAP50']} | mAP50-95: {result['mAP50-95']}")
    print(f"Precision: {result['Precision']} | Recall: {result['Recall']}")
    print(f"Size: {result['Size_MB']} MB | FPS: {result['FPS']}")
    
    return result

In [15]:
all_results = []

for weights in MODELS:
    try:
        res = train_one_model(weights)
        all_results.append(res)
    except Exception as e:
        print(f"❌ Error during training {weights}: {e}")

df_results = pd.DataFrame(all_results)
df_results.to_csv('/kaggle/working/Final_Models_Comparison.csv', index=False)
print(df_results.to_string(index=False))


🚀 Start training yolov8n 🚀
Ultralytics 8.4.94 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_20260713_213825, nbs=64, nms=False, opset=None, optim

KeyboardInterrupt: 

In [ ]:
if not df_results.empty:
    
    best_model_row = df_results.loc[df_results['mAP50'].idxmax()]
    best_weights_source = best_model_row['Weights_Path']
    
    
    best_weights_dest = '/kaggle/working/best_overall_model.pt'
    shutil.copy(best_weights_source, best_weights_dest)
    
    print(f"\n best model: {best_model_row['Model']} acc {best_model_row['mAP50']}")
  
    
    
    
    shutil.make_archive('/kaggle/working/RoadDamage_Project_Files', 'zip', '/kaggle/working/')
    